In [ ]:
import numpy as np
import pandas as pd
import glob
import os
import pprint

# --- Environment and Buffer Constants ---
T_AMB_K = 298.0
P_AMB_BAR = 1.0
U = 64  # Unroll length (new steps per window)
B = 32  # Burn-in length (overlap)

# --- State Indices (Observation) ---
# new_state = (vel_target, self.vel, self.mf, self.brk, self.ice_sp)
IDX_MF = 2
IDX_BRK = 3
IDX_ICE_SP = 4

def load_buffers_from_disk(version_name: str) -> dict:
    """
    Loads all .npz buffer files of a specific version into a dictionary.
    (Function from previous response).
    """
    base_dir = "results"
    buffer_dir = os.path.join(base_dir, version_name, "buffer")
    file_pattern = os.path.join(buffer_dir, "part_*.npz")
    file_paths = glob.glob(file_pattern)
    
    if not file_paths:
        print(f"Warning: No buffer files found in: {buffer_dir}")
        return {}

    print(f"Found {len(file_paths)} buffer files. Loading...")
    buffers_in_memory = {}
    
    for file_path in file_paths:
        try:
            with np.load(file_path) as data:
                # data.files = ['s', 'a', 'r', 'ns', 't', 'tr']
                content = {key: data[key] for key in data.files}
                file_name = os.path.basename(file_path)
                buffers_in_memory[file_name] = content
        except Exception as e:
            print(f"Error loading file {file_path}: {e}")

    print(f"Load complete. {len(buffers_in_memory)} buffers loaded.")
    return buffers_in_memory

def process_buffers_for_models(buffers_in_memory: dict, B_len: int) -> dict:
    """
    Takes the dictionary of buffers, filters episodes that did not reach
    1200 steps, reconstructs trajectories, and extracts inputs for
    ICE and PG models.
    """
    print("Processing data for ICE and PG models...")
    processed_data = {}
    
    discarded_episodes = 0
    
    for file_name, data in buffers_in_memory.items():
        
        # --- 1. FILTERING ---
        # If 'terminated' (data['t']) is True at any point,
        # the episode ended early and was not 'truncated'. We discard it.
        if np.any(data['t']):
            # print(f"  - Discarding {file_name} (ended early).")
            discarded_episodes += 1
            continue
            
        # --- 2. RECONSTRUCTION (Stitching) ---
        # If we are here, the episode was (most likely) 'truncated'
        s_data = data['s']  # Shape (N_windows, S, Obs_dim)
        n_windows = s_data.shape[0]

        if n_windows == 0:
            continue

        # Start with the first complete window
        reconstructed_s = s_data[0, :, :]
        
        # Add only the 'U' (unroll) part of the remaining windows
        for i in range(1, n_windows):
            # Take only the new part, [B:, :]
            unroll_s = s_data[i, B_len:, :]
            reconstructed_s = np.concatenate((reconstructed_s, unroll_s), axis=0)
        
        # NOTE: Reconstructed length will not be 1200.
        # Due to the 'bug' in collect_windows (which does not save the end),
        # for an episode of 1200 steps, N_windows=18.
        # Length = S + 17*U = 96 + 17*64 = 1184 steps.
        # This is the maximum we can recover.

        # --- 3. PROCESSING TO INPUT FORMAT ---
        ice_inputs_list = []
        pg_inputs_list = []
        
        # Iterate through the RECONSTRUCTED trajectory
        for j in range(reconstructed_s.shape[0]):
            s_step = reconstructed_s[j, :]
            
            mf = s_step[IDX_MF]
            brk = s_step[IDX_BRK]
            ice_sp = s_step[IDX_ICE_SP]
            
            # (self.ice_sp, self.mf, self.T_amb_K, self.p_amb_bar)
            ice_tuple = (ice_sp, mf, T_AMB_K, P_AMB_BAR)
            ice_inputs_list.append(ice_tuple)
            
            # (self.ice_sp, 0.0, self.torque_ICE, self.brk)
            pg_tuple = (ice_sp, 0.0, np.nan, brk) 
            pg_inputs_list.append(pg_tuple)

        # Save tuple lists in the output dictionary
        processed_data[file_name] = {
            'ICE_inputs': ice_inputs_list,
            'PG_inputs_parciales': pg_inputs_list
        }
        # print(f"  - Processed {file_name}: {len(ice_inputs_list)} timesteps reconstructed.")

    print(f"Processing completed. {len(processed_data)} episodes processed.")
    print(f"Discarded {discarded_episodes} episodes (ended early).")
    return processed_data

def export_to_csv(processed_data: dict, version_name: str):
    """
    Takes the dictionary of processed data and saves them to CSV files,
    one for each reconstructed episode.
    """
    output_dir = os.path.join("results", version_name, "csv_export_reconstructed")
    os.makedirs(output_dir, exist_ok=True)
    print(f"Exporting CSVs to: {output_dir}")
    
    files_created = 0
    for file_name, data in processed_data.items():
        if not data['ICE_inputs']:
            continue
            
        df_ice = pd.DataFrame(
            data['ICE_inputs'],
            columns=['ICE_in_ice_sp', 'ICE_in_mf', 'ICE_in_T_amb_K', 'ICE_in_p_amb_bar']
        )
        
        df_pg = pd.DataFrame(
            data['PG_inputs_parciales'],
            columns=['PG_in_ice_sp', 'PG_in_em2_torque_soll', 'PG_in_torque_ICE', 'PG_in_brk']
        )
        
        df_final = pd.concat([df_ice, df_pg], axis=1)
        
        csv_file_name = file_name.replace(".npz", ".csv")
        output_path = os.path.join(output_dir, csv_file_name)
        
        df_final.to_csv(output_path, index=False, na_rep='NaN')
        files_created += 1

    print(f"CSV export completed. {files_created} files created.")


# --- USAGE EXAMPLE ---
if __name__ == "__main__":
    
    # 1. Define the version of your model you want to process
    VERSION_TO_LOAD = "pls5"  # <-- CHANGE THIS to your 'version'
    
    # 2. Load buffers from disk to memory
    all_buffers = load_buffers_from_disk(VERSION_TO_LOAD)
    
    if all_buffers:
        # 3. Process buffers: filter, reconstruct and extract
        data_for_models = process_buffers_for_models(all_buffers, B_len=B)
        
        # (Optional) Print a summary of what was processed
        if data_for_models:
            print("\n--- Processing Summary ---")
            first_file = list(data_for_models.keys())[0]
            print(f"Data for first processed episode ('{first_file}'):")
            print(f"  Total reconstructed timesteps: {len(data_for_models[first_file]['ICE_inputs'])}")
            print("  First 5 ICE inputs:")
            pprint.pprint(data_for_models[first_file]['ICE_inputs'][:5])
            print("  First 5 PG inputs:")
            pprint.pprint(data_for_models[first_file]['PG_inputs_parciales'][:5])
        
        # 4. Export the processed dictionary to CSV files
        export_to_csv(data_for_models, VERSION_TO_LOAD)
        
        print("\nProcess finished!")

Encontrados 143 archivos de buffer. Cargando...
Carga completa. 143 buffers cargados.
Procesando datos para modelos ICE y PG...
Procesamiento completado. 143 episodios procesados.
Se descartaron 0 episodios (terminaron antes de tiempo).

--- Resumen del Procesamiento ---
Datos del primer episodio procesado ('part_000093.npz'):
  Total de timesteps reconstruidos: 1184
  Primeros 5 inputs de ICE:
[(2500.0, 50.0, 298.0, 1.0),
 (3326.7947, 42.898167, 298.0, 1.0),
 (3890.3945, 23.327265, 298.0, 1.0),
 (3760.23, 18.413124, 298.0, 1.0),
 (4259.2837, 16.255102, 298.0, 1.0)]
  Primeros 5 inputs de PG:
[(2500.0, 0.0, nan, 0.0),
 (3326.7947, 0.0, nan, 10.2857),
 (3890.3945, 0.0, nan, 9.433094),
 (3760.23, 0.0, nan, 10.385951),
 (4259.2837, 0.0, nan, 4.253569)]
Exportando CSVs a: results/pls5/csv_export_reconstruido
Exportación a CSV completada. 143 archivos creados.

¡Proceso finalizado!


In [ ]:
import numpy as np
import pandas as pd
import glob
import os
import pprint

# --- Environment Constants ---
# These are the constants from your Environment class
T_AMB_K = 298.0
P_AMB_BAR = 1.0

# --- State Indices (Observation) ---
# Based on new_state = (vel_target, self.vel, self.mf, self.brk, self.ice_sp)
IDX_MF = 2
IDX_BRK = 3
IDX_ICE_SP = 4

def load_buffers_from_disk(version_name: str) -> dict:
    """
    Loads all .npz buffer files of a specific version into a dictionary.
    (Function from previous response).
    """
    base_dir = "results"
    buffer_dir = os.path.join(base_dir, version_name, "buffer")
    file_pattern = os.path.join(buffer_dir, "part_*.npz")
    file_paths = glob.glob(file_pattern)
    
    if not file_paths:
        print(f"Warning: No buffer files found in: {buffer_dir}")
        return {}

    print(f"Found {len(file_paths)} buffer files. Loading...")
    buffers_in_memory = {}
    
    for file_path in file_paths:
        try:
            with np.load(file_path) as data:
                content = {key: data[key] for key in data.files}
                file_name = os.path.basename(file_path)
                buffers_in_memory[file_name] = content
        except Exception as e:
            print(f"Error loading file {file_path}: {e}")

    print(f"Load complete. {len(buffers_in_memory)} buffers loaded.")
    return buffers_in_memory

def process_buffers_for_models(buffers_in_memory: dict) -> dict:
    """
    Takes the dictionary of loaded buffers and extracts inputs for
    ICE and PG models.
    """
    print("Processing data for ICE and PG models...")
    processed_data = {}
    
    for file_name, data in buffers_in_memory.items():
        # 's' has shape (N_windows, S_seq_len, Obs_dim)
        states = data['s']
        
        # Lists to save *all* timesteps of this file
        ice_inputs_list = []
        pg_inputs_list = []
        
        num_windows, seq_len, _ = states.shape
        
        # Iterate through each window and each timestep in that window
        for i in range(num_windows):
            for j in range(seq_len):
                # s_step is an array of one timestep (vel_target, vel, mf, brk, ice_sp)
                s_step = states[i, j, :]
                
                # Extract necessary data from state
                mf = s_step[IDX_MF]
                brk = s_step[IDX_BRK]
                ice_sp = s_step[IDX_ICE_SP]
                
                # 1. Create input tuple for ICE
                # (self.ice_sp, self.mf, self.T_amb_K, self.p_amb_bar)
                ice_tuple = (ice_sp, mf, T_AMB_K, P_AMB_BAR)
                ice_inputs_list.append(ice_tuple)
                
                # 2. Create input tuple for PG (partial)
                # (self.ice_sp, 0.0, self.torque_ICE, self.brk)
                # Use np.nan to mark missing data (torque_ICE)
                pg_tuple = (ice_sp, 0.0, np.nan, brk) 
                pg_inputs_list.append(pg_tuple)

        # Save tuple lists in output dictionary
        processed_data[file_name] = {
            'ICE_inputs': ice_inputs_list,
            'PG_inputs_parciales': pg_inputs_list
        }
#         print(f"  - Processed {file_name}: {len(ice_inputs_list)} timesteps.")

    print("Processing completed.")
    return processed_data

def export_to_csv(processed_data: dict, version_name: str):
    """
    Takes the dictionary of processed data and saves them to CSV files,
    one for each original .npz file.
    """
    # Create a directory for CSVs
    output_dir = os.path.join("results", version_name, "csv_export_for_tests")
    os.makedirs(output_dir, exist_ok=True)
    print(f"Exporting CSVs to: {output_dir}")

    for file_name, data in processed_data.items():
        # Create a pandas DataFrame for each model
        df_ice = pd.DataFrame(
            data['ICE_inputs'],
            columns=['ICE_in_ice_sp', 'ICE_in_mf', 'ICE_in_T_amb_K', 'ICE_in_p_amb_bar']
        )
        
        df_pg = pd.DataFrame(
            data['PG_inputs_parciales'],
            columns=['PG_in_ice_sp', 'PG_in_em2_torque_soll', 'PG_in_torque_ICE', 'PG_in_brk']
        )
        
        # Combine the two DataFrames side by side
        df_final = pd.concat([df_ice, df_pg], axis=1)
        
        # Create output file name
        csv_file_name = file_name.replace(".npz", ".csv")
        output_path = os.path.join(output_dir, csv_file_name)
        
        # Save to CSV
        df_final.to_csv(output_path, index=False, na_rep='NaN') # Saves np.nan as 'NaN'

    print(f"CSV export completed. {len(processed_data)} files created.")


# --- USAGE EXAMPLE ---
if __name__ == "__main__":
    
    # 1. Define the version of your model to process
    VERSION_TO_LOAD = "pls5"  # <-- CHANGE THIS to your 'version'
    
    # 2. Load buffers from disk to memory
    all_buffers = load_buffers_from_disk(VERSION_TO_LOAD)
    
    if all_buffers:
        # 3. Process buffers to extract inputs
        data_for_models = process_buffers_for_models(all_buffers)
        
        # (Optional) Print a summary of what was processed
        print("\n--- Processing Summary ---")
        first_file = list(data_for_models.keys())[0]
        print(f"Data for first file ('{first_file}'):")
        pprint.pprint(data_for_models[first_file]['ICE_inputs'][:5])
        pprint.pprint(data_for_models[first_file]['PG_inputs_parciales'][:5])
        
        # 4. Export processed dictionary to CSV files
        export_to_csv(data_for_models, VERSION_TO_LOAD)
        
        print("\nProcess finished!")

Encontrados 143 archivos de buffer. Cargando...
Carga completa. 143 buffers cargados.
Procesando datos para modelos ICE y PG...
Procesamiento completado.

--- Resumen del Procesamiento ---
Datos del primer archivo ('part_000093.npz'):
[(2500.0, 50.0, 298.0, 1.0),
 (3326.7947, 42.898167, 298.0, 1.0),
 (3890.3945, 23.327265, 298.0, 1.0),
 (3760.23, 18.413124, 298.0, 1.0),
 (4259.2837, 16.255102, 298.0, 1.0)]
[(2500.0, 0.0, nan, 0.0),
 (3326.7947, 0.0, nan, 10.2857),
 (3890.3945, 0.0, nan, 9.433094),
 (3760.23, 0.0, nan, 10.385951),
 (4259.2837, 0.0, nan, 4.253569)]
Exportando CSVs a: results/pls5/csv_export_para_pruebas
Exportación a CSV completada. 143 archivos creados.

¡Proceso finalizado!
